# 1. Exploratory Data Analysis & Data Quality Investigation

## Objective & Strategy

Before conducting standard statistical analysis or answering domain-specific EDA questions, we first perform a structured **Data Quality and Sanity Audit**. Geospatial and sensor-derived datasets can contain anomalies such as invalid coordinates, zero-coordinate defaults, corrupted distance measurements, and unrealistic fare values. Performing visualizations or statistical analysis directly on raw data may therefore produce skewed distributions and misleading insights.

Our diagnostic strategy follows a two-tier verification process:

### Tier 1: Global Health Audit
Assess the overall structure and integrity of the dataset by examining:
- Column data types
- Missing-value patterns
- Duplicate records
- Numerical summary statistics
- Minimum and maximum values across numerical features

### Tier 2: Targeted Granular Diagnostics

#### Target Variable (`fare_amount`)
Audit the target variable for:
- Negative fare values
- Zero or suspiciously low fares
- Unrealistic fare values and extreme outliers
- Consistency with the NYC Taxi & Limousine Commission (TLC) base fare, where applicable

#### Geospatial Coordinates
Validate the geographic features by checking:
- Coordinate ranges and expected NYC bounding boxes
- Potential degree/radian representation issues
- Null-Island artifacts such as `(0, 0)`
- Missing or truncated coordinate values

#### Distance Metrics & Consistency
Cross-check distance-related features, including:
- `jfk_dist`
- `lga_dist`
- `ewr_dist`
- `sol_dist`
- `nyc_dist`
- `road_distance_km`

The objective is to identify physically impossible distances, extreme outliers, and inconsistencies between coordinate-based and reported distance measurements.

#### Directional Bearing
Verify the valid range and representation of directional features, particularly whether bearings are expressed as:
- `[-π, π]` radians
- `[0, 2π]` radians
- `[0°, 360°]` degrees

#### Capacity & Categorical Features
Audit:
- Passenger counts for physically implausible values
- Categorical distributions
- Unexpected categories
- Encoding, capitalization, or casing inconsistencies

#### Temporal Features
Validate the logical ranges and consistency of calendar-related features, including:
- Hour
- Day
- Month
- Weekday
- Year

In [47]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [48]:
df = pd.read_csv('../data/processed/uber_with_road_distances.csv')
df.sample(6)

,user_id,user_name,driver_name,car_condition,weather,traffic_condition,key,fare_amount,pickup_datetime,pickup_longitude,...,year,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,distance,bearing,road_distance_km,estimated_duration_min
494679,CYw3rbS8,Brian Berger,Jonathan Morgan,Good,sunny,Flow Traffic,2015-06-24 22:59:11.0000005,25.0,2015-06-24 22:59:11,-1.291714,...,2015,42.518181,33.979317,20.742390,16.313256,9.082202,9.082128,-0.530225,12.1954,19.300000
347915,ZNnkD3az,Jeffrey Jackson,Ryan Jones,Good,cloudy,Dense Traffic,2014-11-18 13:30:56.0000005,6.0,2014-11-18 13:30:56,-1.291335,...,2014,43.108942,33.214876,20.209506,16.864997,8.903432,0.651808,-0.105906,1.7977,4.166667
423235,AvWqJt3d,Thomas Brady,Julie Hurst,Bad,sunny,Congested Traffic,2013-01-12 01:37:28.0000001,9.0,2013-01-12 01:37:28,-1.291243,...,2013,43.282941,34.812341,19.000463,18.763691,10.772132,2.348343,-3.076906,3.5644,5.366667
167927,2SgkqyZM,Annette Burch,Jessica Davis,Bad,cloudy,Flow Traffic,2012-03-29 19:19:41.0000005,5.7,2012-03-29 19:19:41,-1.291576,...,2012,43.602675,31.456927,21.931442,15.051862,7.322361,1.755158,-0.601672,2.0401,4.498333
327932,kNK3P5GG,Jennifer Hill,Nancy Barton,Bad,cloudy,Flow Traffic,2010-12-07 12:15:00.000000186,8.9,2010-12-07 12:15:00,-1.290978,...,2010,43.282798,35.767859,17.996705,19.897205,11.928612,1.705209,1.619150,2.3733,3.918333
298602,AnnOZCzb,Todd Wallace,Patricia Davies,Bad,stormy,Flow Traffic,2014-05-19 02:45:38.0000003,4.5,2014-05-19 02:45:38,-1.290917,...,2014,41.708059,37.005747,16.523765,20.411598,12.148161,0.721110,1.887828,1.1137,2.300000


In [49]:
columns_to_drop = [
    "user_id",
    "user_name",
    "driver_name",
    "key",
    "pickup_datetime",
    "distance",
    "estimated_duration_min"
]

df = df.drop(columns=columns_to_drop)

In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   car_condition      500000 non-null  str    
 1   weather            500000 non-null  str    
 2   traffic_condition  500000 non-null  str    
 3   fare_amount        500000 non-null  float64
 4   pickup_longitude   500000 non-null  float64
 5   pickup_latitude    500000 non-null  float64
 6   dropoff_longitude  499995 non-null  float64
 7   dropoff_latitude   499995 non-null  float64
 8   passenger_count    500000 non-null  int64  
 9   hour               500000 non-null  int64  
 10  day                500000 non-null  int64  
 11  month              500000 non-null  int64  
 12  weekday            500000 non-null  int64  
 13  year               500000 non-null  int64  
 14  jfk_dist           499995 non-null  float64
 15  ewr_dist           499995 non-null  float64
 16  lga_dist     

In [64]:
# 1. Data Quality Report
quality_report = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
    "unique": df.nunique(),
    "dtype": df.dtypes
})

display(quality_report)
display(df.describe().T)

,missing,missing_pct,unique,dtype
car_condition,0,0.0000,4,str
weather,0,0.0000,5,str
traffic_condition,0,0.0000,3,str
fare_amount,0,0.0000,1688,float64
pickup_longitude,0,0.0000,95037,float64
pickup_latitude,0,0.0000,119115,float64
dropoff_longitude,5,0.0010,108283,float64
dropoff_latitude,5,0.0010,135053,float64
passenger_count,0,0.0000,7,int64
hour,0,0.0000,24,int64


,count,mean,std,min,25%,50%,75%,max
fare_amount,500000.0,11.358361,9.916617,-44.900000,6.000000,8.500000,12.500000,500.000000
pickup_longitude,500000.0,-1.265712,0.206941,-52.119764,-1.291405,-1.291226,-1.290970,37.360538
pickup_latitude,500000.0,0.696740,0.140909,-54.389440,0.710958,0.711268,0.711520,29.724576
dropoff_longitude,499995.0,-1.265755,0.205903,-59.049665,-1.291393,-1.291197,-1.290908,0.712985
dropoff_latitude,499995.0,0.696675,0.128997,-44.676047,0.710943,0.711277,0.711538,7.061893
passenger_count,500000.0,1.683428,1.307395,0.000000,1.000000,1.000000,2.000000,6.000000
hour,500000.0,13.510834,6.511571,0.000000,9.000000,14.000000,19.000000,23.000000
day,500000.0,15.684206,8.681066,1.000000,8.000000,16.000000,23.000000,31.000000
month,500000.0,6.268650,3.437815,1.000000,3.000000,6.000000,9.000000,12.000000
weekday,500000.0,3.042008,1.949240,0.000000,1.000000,3.000000,5.000000,6.000000


In [65]:
# 2. Passenger Count Investigation
print("--- PASSENGER COUNT VALUE COUNTS ---")
print(df["passenger_count"].value_counts(dropna=False).sort_index())

--- PASSENGER COUNT VALUE COUNTS ---
passenger_count
0      1796
1    346009
2     73908
3     21761
4     10614
5     35322
6     10590
Name: count, dtype: int64


In [67]:
# 3. Distances Investigation
distance_cols = [
    "jfk_dist",
    "ewr_dist",
    "lga_dist",
    "sol_dist",
    "nyc_dist",
    "road_distance_km"
]

print("--- DISTANCES SUMMARY ---")
display(df[distance_cols].describe().T)

print("\n--- SANITY CHECKS (Negative / Zero / Missing) ---")
for col in distance_cols:
    print(f"\n{col}:")
    print("Negative:", (df[col] < 0).sum())
    print("Zero:", (df[col] == 0).sum())
    print("Missing:", df[col].isna().sum())

--- DISTANCES SUMMARY ---


,count,mean,std,min,25%,50%,75%,max
jfk_dist,499995.0,385.279367,2419.087483,1.017646,41.341514,42.523163,43.785649,30133.067880
ewr_dist,499995.0,380.503657,2428.804740,1.460945,32.173712,34.787507,38.304502,30167.595967
lga_dist,499995.0,363.843772,2425.075903,0.382119,17.100762,19.591554,22.214815,30167.285794
sol_dist,499995.0,363.674038,2428.348683,0.532545,14.886989,18.347580,22.417812,30159.407296
nyc_dist,499995.0,355.991423,2428.730839,0.080500,7.147384,10.458151,14.448699,30162.285356
road_distance_km,486674.0,4.394519,4.766610,0.000000,1.708400,2.837000,5.063375,154.878300



--- SANITY CHECKS (Negative / Zero / Missing) ---

jfk_dist:
Negative: 0
Zero: 0
Missing: 5

ewr_dist:
Negative: 0
Zero: 0
Missing: 5

lga_dist:
Negative: 0
Zero: 0
Missing: 5

sol_dist:
Negative: 0
Zero: 0
Missing: 5

nyc_dist:
Negative: 0
Zero: 0
Missing: 5

road_distance_km:
Negative: 0
Zero: 5035
Missing: 13326


In [68]:
# 4. Coordinates Investigation
coord_cols = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude"
]

print("--- COORDINATES SUMMARY ---")
display(df[coord_cols].describe().T)

print("\n--- COORDINATES SANITY CHECKS ---")
for col in coord_cols:
    print(f"\n{col}:")
    print("Zero:", (df[col] == 0).sum())
    print("Missing:", df[col].isna().sum())
    print("Min:", df[col].min(), "Max:", df[col].max())

--- COORDINATES SUMMARY ---


,count,mean,std,min,25%,50%,75%,max
pickup_longitude,500000.0,-1.265712,0.206941,-52.119764,-1.291405,-1.291226,-1.290970,37.360538
pickup_latitude,500000.0,0.696740,0.140909,-54.389440,0.710958,0.711268,0.711520,29.724576
dropoff_longitude,499995.0,-1.265755,0.205903,-59.049665,-1.291393,-1.291197,-1.290908,0.712985
dropoff_latitude,499995.0,0.696675,0.128997,-44.676047,0.710943,0.711277,0.711538,7.061893



--- COORDINATES SANITY CHECKS ---

pickup_longitude:
Zero: 9432
Missing: 0
Min: -52.11976380072031 Max: 37.36053821400994

pickup_latitude:
Zero: 9408
Missing: 0
Min: -54.389440365122525 Max: 29.72457633831709

dropoff_longitude:
Zero: 9455
Missing: 5
Min: -59.049665381155926 Max: 0.7129849239711017

dropoff_latitude:
Zero: 9426
Missing: 5
Min: -44.67604655619588 Max: 7.061893047595487


In [69]:
# 5. Fare Amount Investigation
print("--- FARE AMOUNT PERCENTILES ---")
display(df["fare_amount"].describe(percentiles=[
    .001, .01, .05, .25, .5, .75, .95, .99, .999
]))

print("\n--- FARE SANITY CHECKS ---")
print("Negative Fares (< 0):", (df["fare_amount"] < 0).sum())
print("Zero Fares (== 0):", (df["fare_amount"] == 0).sum())
print("Fares <= 0:", (df["fare_amount"] <= 0).sum())
print("Min Fare:", df["fare_amount"].min())
print("Max Fare:", df["fare_amount"].max())

--- FARE AMOUNT PERCENTILES ---


count    500000.000000
mean         11.358361
std           9.916617
min         -44.900000
0.1%          2.500000
1%            3.300000
5%            4.100000
25%           6.000000
50%           8.500000
75%          12.500000
95%          30.500000
99%          52.101000
99.9%        80.000000
max         500.000000
Name: fare_amount, dtype: float64


--- FARE SANITY CHECKS ---
Negative Fares (< 0): 21
Zero Fares (== 0): 14
Fares <= 0: 35
Min Fare: -44.9
Max Fare: 500.0


In [70]:
# 6. Categorical Features Investigation
categorical_cols = ["car_condition", "weather", "traffic_condition"]

for col in categorical_cols:
    print(f"\n--- {col} VALUE COUNTS ---")
    print(df[col].value_counts(dropna=False))


--- car_condition VALUE COUNTS ---
car_condition
Very Good    125312
Bad          124978
Good         124968
Excellent    124742
Name: count, dtype: int64

--- weather VALUE COUNTS ---
weather
sunny     100433
cloudy    100062
rainy      99972
stormy     99955
windy      99578
Name: count, dtype: int64

--- traffic_condition VALUE COUNTS ---
traffic_condition
Congested Traffic    166847
Dense Traffic        166584
Flow Traffic         166569
Name: count, dtype: int64


In [71]:
# 7. Time Features Sanity Checks
time_cols = ["hour", "day", "month", "weekday", "year"]

print("--- TIME FEATURES RANGES ---")
for col in time_cols:
    print(f"{col}: min = {df[col].min()}, max = {df[col].max()}, unique = {df[col].nunique()}, missing = {df[col].isna().sum()}")

--- TIME FEATURES RANGES ---
hour: min = 0, max = 23, unique = 24, missing = 0
day: min = 1, max = 31, unique = 31, missing = 0
month: min = 1, max = 12, unique = 12, missing = 0
weekday: min = 0, max = 6, unique = 7, missing = 0
year: min = 2009, max = 2015, unique = 7, missing = 0


In [72]:
# 8. Bearing & Duplicate/Consistency Checks
print("--- BEARING INVESTIGATION ---")
display(df["bearing"].describe())
print("Bearing < 0:", (df["bearing"] < 0).sum())
print("Bearing > 360:", (df["bearing"] > 360).sum())
print("Bearing Missing:", df["bearing"].isna().sum())

print("\n--- DUPLICATES CHECK ---")
print("Full Row Duplicates:", df.duplicated().sum())

same_coords = (
    (df["pickup_longitude"] == df["dropoff_longitude"]) &
    (df["pickup_latitude"] == df["dropoff_latitude"])
)
print("\nTrips with exactly identical pickup and dropoff coords:", same_coords.sum())

--- BEARING INVESTIGATION ---


count    499995.000000
mean          0.297145
std           1.804548
min          -3.141593
25%          -0.854721
50%          -0.050442
75%           2.206769
max           3.141593
Name: bearing, dtype: float64

Bearing < 0: 254223
Bearing > 360: 0
Bearing Missing: 5

--- DUPLICATES CHECK ---
Full Row Duplicates: 0

Trips with exactly identical pickup and dropoff coords: 14250


# 2. Data Quality Audit Findings & Preprocessing Strategy

## Diagnostic Findings & Structural Anomalies Detected

Following the execution of the sanity audit across all 21 feature columns (500,000 observations), we identified the following critical anomalies:

1. **Spatial Coordinates & Bearing Unit Shift (Radians vs. Degrees):**
   - Coordinates are stored in **radians**, not standard geographic decimal degrees. Median longitude is $-1.2912 \text{ rad} \approx -73.98^\circ \text{ W}$ and median latitude is $0.7112 \text{ rad} \approx 40.75^\circ \text{ N}$, corresponding exactly to the New York metropolitan area.
   - Bearing is bounded precisely between $-\pi$ and $+\pi$ ($[-3.141593, 3.141593]$), representing the raw output of an $\text{atan2}(y, x)$ transformation rather than a compass azimuth $[0, 360^\circ)$.

2. **Geographic Corruption & Null-Island Artifacts:**
   - Approximately 9,400+ entries contain coordinate pairs locked at $(0, 0)$ or extreme coordinates (e.g., latitude $-54.38$), placing origins deep in the Atlantic Ocean or South America.
   - Landmark distance metrics (`jfk_dist`, `nyc_dist`, etc.) peak at $30,167 \text{ km}$ strictly due to Euclidean/Spherical calculations computed from these corrupted coordinates.
   - 14,250 trips record identical pickup and dropoff points (zero net displacement).

3. **Missing Value Topology:**
   - **5 Shared Null Rows:** Columns `dropoff_longitude`, `dropoff_latitude`, all 5 landmark distances, and `bearing` share the identical 5 missing records. Dropping these coordinates cascaded into distance calculation failures.
   - **Road Distance Deficit:** `road_distance_km` contains 13,326 missing entries ($\approx 2.67\%$) alongside 5,035 zero-distance recordings.

4. **Fare Anomalies & Capacity Verification:**
   - `fare_amount` includes 21 negative fares (down to $-\$44.90$) and 14 zero-dollar transactions. 99.9% of valid trips sit below $\$80.00$, with an absolute cap at $\$500.00$.
   - `passenger_count` contains 1,796 records with 0 passengers; all other counts (1 to 6) fit standard commercial vehicle capacities.
   - `time_features` and `categorical_features` are completely clean, balanced, and error-free.

---

## Preprocessing Decision Rules

Based on defensible business logic and spatial constraints, we enforce the following cleaning pipeline:

| Target Domain | Cleaning Action | Justification / Technical Threshold |
| :--- | :--- | :--- |
| **Cascade Nulls** | Drop the 5 null rows | Impact is negligible ($0.001\%$), eliminates cascading missing values across 8 columns. |
| **Spatial Bounding** | Filter coordinates to Greater NYC | Bound radians to $[-1.300, -1.280]$ lon and $[0.700, 0.720]$ lat. Purges $(0,0)$ artifacts and astronomical landmark distances. |
| **Fare Bounds** | Enforce $\$2.50 \le \text{fare} \le \$250.00$ | Eliminates negative/zero fares below legal NYC TLC minimum drop rate while truncating extreme unverified outliers. |
| **Passenger Count** | Impute or filter $0$ passengers | Passenger count of 0 is physically invalid for standard passenger transport; align to baseline (1) or prune. |
| **Bearing Alignment** | Map radians to $[0, 360^\circ)$ | Convert via: $\text{bearing\_deg} = (\text{np.degrees}(\text{bearing}) + 360) \pmod{360}$. |
| **Missing Distance** | Impute/compute spatial distance | Calculate Haversine distance from validated coordinates to fill missing or zero values in `road_distance_km`. |

---

In [73]:
# 1. Create a working copy
df_clean = df.copy()

initial_rows = len(df_clean)
print(f"Initial row count: {initial_rows:,}")

# 2. Drop the 5 shared missing coordinate rows
df_clean = df_clean.dropna(subset=["dropoff_longitude", "dropoff_latitude"])
dropped_nulls = initial_rows - len(df_clean)
print(f"Rows after dropping coordinate nulls: {len(df_clean):,} (Dropped: {dropped_nulls})")

# 3. Define Greater NYC geographic boundary box in radians
# Approximate equivalent in degrees: Longitude [-74.5, -72.8], Latitude [40.4, 41.2]
LON_MIN, LON_MAX = np.radians(-74.5), np.radians(-72.8)
LAT_MIN, LAT_MAX = np.radians(40.4), np.radians(41.2)

geo_mask = (
    df_clean["pickup_longitude"].between(LON_MIN, LON_MAX) &
    df_clean["dropoff_longitude"].between(LON_MIN, LON_MAX) &
    df_clean["pickup_latitude"].between(LAT_MIN, LAT_MAX) &
    df_clean["dropoff_latitude"].between(LAT_MIN, LAT_MAX)
)

dropped_geo = len(df_clean) - geo_mask.sum()
df_clean = df_clean[geo_mask].reset_index(drop=True)
print(f"Rows after NYC spatial bounding: {len(df_clean):,} (Dropped invalid coords: {dropped_geo:,})")

# 4. Verify eradication of corrupted astronomical landmark distances
distance_cols = ["jfk_dist", "ewr_dist", "lga_dist", "sol_dist", "nyc_dist"]
print("\n--- Max Landmark Distances After Spatial Filter ---")
print(df_clean[distance_cols].max())

Initial row count: 500,000
Rows after dropping coordinate nulls: 499,995 (Dropped: 5)
Rows after NYC spatial bounding: 489,319 (Dropped invalid coords: 10,676)

--- Max Landmark Distances After Spatial Filter ---
jfk_dist    156.212265
ewr_dist    222.699037
lga_dist    176.480475
sol_dist    201.801801
nyc_dist    196.084719
dtype: float64


In [74]:
# 1. Filter fare_amount based on NYC TLC base fare rules and realistic upper bound
fare_mask = (df_clean["fare_amount"] >= 2.5) & (df_clean["fare_amount"] <= 250.0)
dropped_fares = len(df_clean) - fare_mask.sum()

df_clean = df_clean[fare_mask].reset_index(drop=True)
print(f"Rows after fare filtering ($2.5 to $250): {len(df_clean):,} (Dropped: {dropped_fares:,})")

# 2. Handle zero passenger counts (impute with median/mode = 1)
zero_passengers_count = (df_clean["passenger_count"] == 0).sum()
print(f"Zero passenger count entries before fix: {zero_passengers_count:,}")

df_clean["passenger_count"] = df_clean["passenger_count"].replace(0, 1)

# 3. Verification summary
print("\n--- FARE AMOUNT SUMMARY ---")
print(df_clean["fare_amount"].describe(percentiles=[.01, .5, .99, .999]))

print("\n--- PASSENGER COUNT SUMMARY ---")
print(df_clean["passenger_count"].value_counts().sort_index())

Rows after fare filtering ($2.5 to $250): 489,273 (Dropped: 46)
Zero passenger count entries before fix: 1,754

--- FARE AMOUNT SUMMARY ---
count    489273.000000
mean         11.332718
std           9.661276
min           2.500000
1%            3.300000
50%           8.500000
99%          52.000000
99.9%        78.750000
max         250.000000
Name: fare_amount, dtype: float64

--- PASSENGER COUNT SUMMARY ---
passenger_count
1    340155
2     72459
3     21316
4     10372
5     34590
6     10381
Name: count, dtype: int64


In [75]:
# 1. Convert bearing from radians [-pi, pi] to standard degrees [0, 360)
df_clean["bearing_deg"] = (np.degrees(df_clean["bearing"]) + 360) % 360

# 2. Compute Haversine distance in km directly from radian coordinates
# Earth radius = 6371.0088 km
def haversine_np(lon1, lat1, lon2, lat2):
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0088 * c

df_clean["haversine_dist_km"] = haversine_np(
    df_clean["pickup_longitude"],
    df_clean["pickup_latitude"],
    df_clean["dropoff_longitude"],
    df_clean["dropoff_latitude"]
)

# 3. Handle road_distance_km missing & zero values
# Calculate the empirical ratio between valid road_distance and haversine_dist
valid_distance_mask = (df_clean["road_distance_km"] > 0) & (df_clean["haversine_dist_km"] > 0.05)
routing_factor = (df_clean.loc[valid_distance_mask, "road_distance_km"] /
                  df_clean.loc[valid_distance_mask, "haversine_dist_km"]).median()

print(f"Empirical Manhattan Routing Factor (Road / Haversine): {routing_factor:.3f}")

# Impute road_distance_km where missing or zero using haversine * routing_factor
missing_or_zero = df_clean["road_distance_km"].isna() | (df_clean["road_distance_km"] <= 0)
print(f"Total road_distance entries imputed: {missing_or_zero.sum():,}")

df_clean["road_distance_km"] = np.where(
    missing_or_zero,
    df_clean["haversine_dist_km"] * routing_factor,
    df_clean["road_distance_km"]
)

# 4. Final verification of processed columns
print("\n--- BEARING DEG SUMMARY ---")
print(df_clean["bearing_deg"].describe())

print("\n--- ROAD DISTANCE SUMMARY ---")
print(df_clean["road_distance_km"].describe())
print(f"Missing in road_distance_km: {df_clean['road_distance_km'].isna().sum()}")

Empirical Manhattan Routing Factor (Road / Haversine): 1.325
Total road_distance entries imputed: 7,680

--- BEARING DEG SUMMARY ---
count    489273.000000
mean        203.893746
std         105.606060
min           0.000000
25%         134.202861
50%         186.369622
75%         315.029935
max         359.998528
Name: bearing_deg, dtype: float64

--- ROAD DISTANCE SUMMARY ---
count    489273.000000
mean          4.441552
std           4.874395
min           0.000000
25%           1.708300
50%           2.842700
75%           5.091700
max         120.859700
Name: road_distance_km, dtype: float64
Missing in road_distance_km: 0


In [76]:
# 1. Final Sanity & Completeness Check
final_report = pd.DataFrame({
    "missing": df_clean.isna().sum(),
    "missing_pct": df_clean.isna().mean() * 100,
    "unique": df_clean.nunique(),
    "dtype": df_clean.dtypes
})

print(f"Final clean dataset shape: {df_clean.shape}")
print(f"Data retention rate: {(len(df_clean) / 500000) * 100:.2f}%")
print("\n--- FINAL MISSING VALUES CHECK ---")
print(f"Total NaN values across entire dataframe: {df_clean.isna().sum().sum()}")

# 2. Display summary stats of cleaned numerical features
display(df_clean.describe().T)

Final clean dataset shape: (489273, 23)
Data retention rate: 97.85%

--- FINAL MISSING VALUES CHECK ---
Total NaN values across entire dataframe: 0


,count,mean,std,min,25%,50%,75%,max
fare_amount,489273.0,11.332718,9.661276,2.500000,6.000000,8.500000,12.500000,250.000000
pickup_longitude,489273.0,-1.291114,0.000617,-1.300174,-1.291409,-1.291231,-1.290991,-1.271594
pickup_latitude,489273.0,0.711236,0.000474,0.705150,0.710987,0.711281,0.711526,0.718799
dropoff_longitude,489273.0,-1.291096,0.000620,-1.299827,-1.291396,-1.291205,-1.290938,-1.271559
dropoff_latitude,489273.0,0.711244,0.000544,0.705190,0.710971,0.711289,0.711543,0.718669
passenger_count,489273.0,1.687698,1.304532,1.000000,1.000000,1.000000,2.000000,6.000000
hour,489273.0,13.511874,6.508116,0.000000,9.000000,14.000000,19.000000,23.000000
day,489273.0,15.679471,8.680673,1.000000,8.000000,16.000000,23.000000,31.000000
month,489273.0,6.270277,3.439732,1.000000,3.000000,6.000000,9.000000,12.000000
weekday,489273.0,3.041991,1.948942,0.000000,1.000000,3.000000,5.000000,6.000000


# 3. Preprocessing Execution & Validation Summary

## Applied Transformations & Cleaning Pipeline
1. **Cascade Null Pruning:** Removed 5 records with missing coordinates and landmark distance metrics.
2. **NYC Bounding Box Enforcement:** Radian coordinates were restricted to Greater NYC bounds:
   - Longitude: $[-1.300, -1.280] \text{ rad} \approx [-74.5^\circ, -72.8^\circ]$
   - Latitude: $[0.700, 0.720] \text{ rad} \approx [40.4^\circ, 41.2^\circ]$
   - Purged 10,676 invalid entries (zero coordinates and coordinate anomalies).
3. **Fare Normalization:** Enforced $\$2.50 \le \text{fare\_amount} \le \$250.00$, dropping 46 invalid rows.
4. **Passenger Imputation:** Imputed 1,754 zero-passenger entries with the modal/median capacity ($1$).
5. **Directional Mapping:** Converted angular bearing from $[-\pi, \pi]$ radians to standard compass degrees $[0, 360^\circ)$ via `bearing_deg`.
6. **Distance Imputation & Routing Factor:**
   - Derived direct spherical distance using vectorised `haversine_dist_km`.
   - Empirically determined the Manhattan grid tortuosity factor:
     $$\text{Routing Factor} = \frac{\text{road\_distance\_km}}{\text{haversine\_dist\_km}} \approx 1.325$$
   - Imputed 7,680 zero/missing `road_distance_km` values using the empirical Manhattan routing multiplier.

## Final Dataset State
- **Clean Rows:** 489,273 / 500,000 (97.85% retention rate).
- **Null Count:** 0 missing values across all 23 columns.
- **Dimensionality:** 23 features ready for exploratory feature analysis and predictive modeling.

In [78]:
import os

# Define output path in data/processed directory
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

clean_file_path = os.path.join(output_dir, "uber_cleaned_data.csv")

# Save cleaned dataframe to CSV
df_clean.to_csv(clean_file_path, index=False)

print(f"Cleaned dataset successfully saved to: {clean_file_path}")
print(f"Shape: {df_clean.shape} | Total memory: {df_clean.memory_usage().sum() / 1e6:.2f} MB")

Cleaned dataset successfully saved to: ../data/processed/uber_cleaned_data.csv
Shape: (489273, 23) | Total memory: 90.03 MB
